# 21 — Class-membership probabilities

This notebook retains the accepted classifier's complete probability vector instead of immediately reducing it to a hard label. Each row receives functional, repair and non-functional membership estimates summing to one.

**Result:** the accepted model already provides the intended memberships. Its out-of-fold top-label calibration error is 0.708%, and the validated 14,850-row competition membership table is stored under the ignored runtime directory.

## Course-aligned lifecycle

| Step | Application |
| --- | --- |
| 1. Define the goal and scope | Retain all three accepted-model probabilities as class memberships. |
| 2. Gather the data | Reuse accepted OOF probabilities, then refit the accepted ensemble for competition inference. |
| 3. Explore the data | Compare memberships, calibration, confidence, margins, entropy and runner-up boundaries. |
| 4. Clean and preprocess the data | Reuse accepted fold-fitted and full-labelled-data preprocessing. |
| 5. Select and engineer features | Keep the accepted feature policy unchanged. |
| 6. Define the machine-learning task | Supervised probabilistic three-class classification. |
| 7. Partition the data | Use OOF probabilities for labelled diagnostics and competition rows only for unlabelled inference. |
| 8. Select and train candidate methods | No new selection; refit the accepted 55:45 XGBoost/Random-Forest vote. |
| 9. Evaluate and interpret the results | Evaluate calibration where labels exist and compare unlabelled membership distributions without claiming test accuracy. |
| 10. Deploy and iterate | Export the membership table, then move the next bounded modelling loop to outlier filtering. |

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display

STAGE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = STAGE_DIR / 'src'
PROJECT_DIR = STAGE_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

result = joblib.load(
    PROJECT_DIR / '.runtime' / 'class-membership-analysis'
    / 'class-membership-analysis.joblib'
)
competition_memberships = pd.read_csv(result['competition_table_path'])
competition_memberships.head()

## Probability is the membership

The accepted ensemble estimates

$$P(y \mid x)=0.55P_{\text{XGBoost}}(y \mid x)+0.45P_{\text{forest}}(y \mid x).$$

The competition label is simply `argmax` of this vector. The export also retains the runner-up, winning margin and normalised entropy, so a prediction such as `(0.58, 0.05, 0.37)` is not misrepresented as unambiguously functional.

In [ ]:
display(result['probability_quality'].to_frame('value'))
display(result['classwise_quality'])
display(result['component_disagreement'].to_frame('value'))

## What the hard repair label hides

Mean competition repair membership is **7.178%**, close to the labelled prevalence, but repair is the maximum-probability class for only **3.953%** of rows. There are 1,323 competition rows with at least 25% repair membership, compared with 587 hard repair predictions.

This is exactly the information lost when the probability vector is reduced to one class. It also reflects a difficult middle class: actual development repair cases receive only 33.766% repair membership on average.

In [ ]:
display(pd.concat({
    'development': result['development_summary'],
    'competition': result['competition_summary'],
}, names=['partition']))
display(pd.concat({
    'development': result['development_ambiguity'],
    'competition': result['competition_ambiguity'],
}, axis=1))

## Ambiguity cohorts

A fixed margin below ten points identifies 2,810 development rows and 875 competition rows. Development accuracy within that cohort is 50.107%. A runner-up membership of at least 25% identifies 26.883% of development and 27.596% of competition, with 59.734% development accuracy.

The overall membership distributions are closely aligned across partitions. That argues against a large aggregate shift but does not establish competition accuracy.

In [ ]:
competition_memberships.sort_values(
    ['winning_margin', 'normalised_entropy'],
    ascending=[True, False],
).head(20)

## Decision and next loop

Use the accepted probability vector directly as class membership. Keep maximum probability for competition labels unless another decision objective is specified and evaluated fold-safely. No new fuzzy-label training is required.

The next modelling loop should now switch to **outlier filtering**. Start with training-only, fold-fitted policies for physically impossible states, multivariate support outliers, isolated categorical/geographic rows and label-conflict candidates. Validation rows must remain untouched, and model uncertainty alone must not be treated as proof of a data outlier.

The full interpretation and reproduction command are in [`../reports/class-membership-probabilities.md`](../reports/class-membership-probabilities.md).